# ChemBreak9 — Adaptive MDP Jailbreak Study
**Research target:** ChemDFM · ChemLLM  
**Condition:** C3\_ADAPTIVE\_MDP only (MDP-driven multi-turn jailbreak)  
**Phases:** development → pilot → full\_bank  

> C0/C1/C2 baselines are excluded from this run. They can be added later with a frozen policy for comparison and do not affect Q-policy training.

In [ ]:
# ── User settings — fill these in before running ──────────────────────────
PROJECT_ID  = "REPLACE_WITH_YOUR_GCP_PROJECT_ID"  # e.g. "my-gcp-project-123"
REPO_URL    = "https://github.com/Jollychuks/ChemBreak.git"
BRANCH      = "main"
PHASE       = "development"  # development | pilot | full_bank
LIVE        = False           # False = dry-run / mock; True = real models

# ── Derived (do not edit) ──────────────────────────────────────────────────
REPO_NAME   = "ChemBreak"
PKG_DIR     = "chembreak9"
CONFIG_PATH = f"{REPO_NAME}/{PKG_DIR}/configs/config.{PHASE}.yaml"
import os
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["CHEMBREAK_ENABLE_LIVE"] = "YES" if LIVE else "NO"
print(f"Phase: {PHASE} | Live: {LIVE} | Project: {PROJECT_ID}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

STORAGE = '/content/chembreak9_storage'
DRIVE_STORAGE = '/content/drive/MyDrive/chembreak9_storage'
!mkdir -p {STORAGE} && [ -d {DRIVE_STORAGE} ] && ln -sf {DRIVE_STORAGE} {STORAGE} || mkdir -p {STORAGE}

# Clone / update repo
import os
if os.path.exists(REPO_NAME):
    !cd {REPO_NAME} && git pull origin {BRANCH} -q
else:
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} -q

# Install package and deps from repo-local requirements
!pip install -q -r {REPO_NAME}/{PKG_DIR}/requirements.txt
!pip install -q -e {REPO_NAME}/{PKG_DIR}
print('Environment ready')

In [ ]:
from chembreak9.preflight import run_preflight
status = run_preflight(CONFIG_PATH)
import json, pprint
pprint.pprint(status)
assert status['status'] == 'ok', 'Preflight failed — fix errors above before continuing'

## C3\_ADAPTIVE\_MDP — ChemDFM

MDP-driven adaptive jailbreak. The Q-policy learns which tactics break ChemDFM's refusals across 48 tasks. Checkpointed every episode — safe to restart if interrupted.

In [ ]:
from chembreak9.runner import run_one
RUN_DIR = run_one(CONFIG_PATH, 'C3_ADAPTIVE_MDP', 'ChemDFM')
print('ChemDFM C3 complete:', RUN_DIR)

## C3\_ADAPTIVE\_MDP — ChemLLM

Same policy, same 48 tasks — now against ChemLLM (InternLM-2 base). The Q-table continues updating from where ChemDFM left off.

In [ ]:
RUN_DIR = run_one(CONFIG_PATH, 'C3_ADAPTIVE_MDP', 'ChemLLM')
print('ChemLLM C3 complete:', RUN_DIR)

## Freeze policy

Lock the trained Q-table before running pilot or full\_bank. A frozen policy no longer updates — all subsequent evaluations measure what the policy learned, not what it is still learning.

In [ ]:
from chembreak9.policy import AdaptiveQPolicy
from chembreak9.config import load_config
cfg = load_config(CONFIG_PATH)
policy = AdaptiveQPolicy.load(cfg['policy']['artifact_path'])
policy.freeze()
policy.save(cfg['policy']['artifact_path'])
print(f'Policy frozen. States: {policy.exact_states} | Updates: {policy.updates}')

## Results

In [ ]:
from chembreak9.metrics import load_release_metrics
import pandas as pd

episodes, metrics = load_release_metrics(RUN_DIR, condition='C3_ADAPTIVE_MDP')

print('=== Episode results ===')
display(episodes)

print('\n=== Metrics summary ===')
display(metrics)

print(f'\nEpisode CSV: {RUN_DIR}/release/episode_results_C3_ADAPTIVE_MDP.csv')
print(f'Metrics CSV: {RUN_DIR}/release/metrics_C3_ADAPTIVE_MDP.csv')

## Download release files

In [ ]:
import shutil, pathlib
from google.colab import files

release_dir = pathlib.Path(RUN_DIR) / 'release'
zip_path = f'/content/CB9_C3_{PHASE}_release.zip'
shutil.make_archive(zip_path.replace('.zip',''), 'zip', release_dir)
files.download(zip_path)
print('Download started')